# Geographical Aggregation (Tourism) — Covariance Method Comparison

> Comparing MinTrace covariance estimation methods on Australian Tourism Data

This notebook extends the [basic Australian Tourism example](australiandomestictourism.ipynb) by comparing **cross-sectional covariance estimation methods** available in `HierarchicalForecast` for MinTrace reconciliation.

The MinTrace reconciliation solution depends on an estimate of the covariance matrix $\mathbf{W}_h$. Different estimators trade off bias, variance, and computational cost:

| Method | Name | Description | Diagonal? | Requires residuals? |
|--------|------|-------------|-----------|---------------------|
| `ols` | OLS | Identity matrix | Yes | No |
| `wls_struct` | WLS (structural) | Scales by number of bottom nodes | Yes | No |
| `wls_var` | WLS (variance) | Per-series variance from residuals | Yes | Yes |
| `shr` / `mint_shrink` | Shrinkage | Ledoit-Wolf shrinkage covariance | No | Yes |
| `oasd` | Oracle Approx. Shrinkage Diagonal | Per-element shrinkage of correlation | No | Yes |

**Note:** The `sam`/`mint_cov` (sample covariance) and `bu` (bottom-up covariance) methods are omitted because they are rank-deficient when the number of series (425) exceeds the number of in-sample observations (72). The `shr`/`mint_shrink` shrinkage estimator regularizes `sam` to handle this case.

We use the same dataset, hierarchy, and base forecasts as the original example.

You can run these experiments using CPU or GPU with Google Colab.

<a href="https://colab.research.google.com/github/Nixtla/hierarchicalforecast/blob/main/nbs/examples/AustralianDomesticTourism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install hierarchicalforecast statsforecast

Defaulting to user installation because normal site-packages is not writeable


## 1. Load and Process Data

In this example we will use the [Tourism](https://otexts.com/fpp3/tourism.html) dataset from the [Forecasting: Principles and Practice](https://otexts.com/fpp3/) book.

The dataset only contains the time series at the lowest level, so we need to create the time series for all hierarchies.

In [11]:
import numpy as np
import pandas as pd

In [12]:
Y_df = pd.read_csv('https://raw.githubusercontent.com/Nixtla/transfer-learning-time-series/main/datasets/tourism.csv')
Y_df = Y_df.rename({'Trips': 'y', 'Quarter': 'ds'}, axis=1)
Y_df.insert(0, 'Country', 'Australia')
Y_df = Y_df[['Country', 'Region', 'State', 'Purpose', 'ds', 'y']]
Y_df['ds'] = Y_df['ds'].str.replace(r'(\d+) (Q\d)', r'\1-\2', regex=True)
Y_df['ds'] = pd.PeriodIndex(Y_df["ds"], freq='Q').to_timestamp()
Y_df.head()

,Country,Region,State,Purpose,ds,y
0,Australia,Adelaide,South Australia,Business,1998-01-01,135.077690
1,Australia,Adelaide,South Australia,Business,1998-04-01,109.987316
2,Australia,Adelaide,South Australia,Business,1998-07-01,166.034687
3,Australia,Adelaide,South Australia,Business,1998-10-01,127.160464
4,Australia,Adelaide,South Australia,Business,1999-01-01,137.448533


The dataset can be grouped in the following non-strictly hierarchical structure.

In [13]:
spec = [
    ['Country'],
    ['Country', 'State'], 
    # ['Country', 'Purpose'], 
    ['Country', 'State', 'Region'], 
    # ['Country', 'State', 'Purpose'], 
    ['Country', 'State', 'Region', 'Purpose']
]

Using the `aggregate` function from `HierarchicalForecast` we can get the full set of time series.

In [14]:
from hierarchicalforecast.utils import aggregate

In [15]:
Y_df, S_df, tags = aggregate(Y_df, spec)

In [16]:
Y_df.head()

,unique_id,ds,y
0,Australia,1998-01-01,23182.197269
1,Australia,1998-04-01,20323.380067
2,Australia,1998-07-01,19826.640511
3,Australia,1998-10-01,20830.129891
4,Australia,1999-01-01,22087.353380


In [17]:
S_df.iloc[:5, :5]

,unique_id,Australia/ACT/Canberra/Business,Australia/ACT/Canberra/Holiday,Australia/ACT/Canberra/Other,Australia/ACT/Canberra/Visiting
0,Australia,1.0,1.0,1.0,1.0
1,Australia/ACT,1.0,1.0,1.0,1.0
2,Australia/New South Wales,0.0,0.0,0.0,0.0
3,Australia/Northern Territory,0.0,0.0,0.0,0.0
4,Australia/Queensland,0.0,0.0,0.0,0.0


### Split Train/Test sets

We use the final two years (8 quarters) as test set.

In [18]:
Y_test_df = Y_df.groupby('unique_id', as_index=False).tail(8)
Y_train_df = Y_df.drop(Y_test_df.index)

In [19]:
Y_train_df.groupby('unique_id').size()

unique_id
Australia                                                72
Australia/ACT                                            72
Australia/ACT/Canberra                                   72
Australia/ACT/Canberra/Business                          72
Australia/ACT/Canberra/Holiday                           72
                                                         ..
Australia/Western Australia/Experience Perth             72
Australia/Western Australia/Experience Perth/Business    72
Australia/Western Australia/Experience Perth/Holiday     72
Australia/Western Australia/Experience Perth/Other       72
Australia/Western Australia/Experience Perth/Visiting    72
Length: 389, dtype: int64

## 2. Computing base forecasts

The following cell computes the **base forecasts** for each time series in `Y_df` using the `ETS` model. Observe that `Y_hat_df` contains the forecasts but they are not coherent.

In [20]:
from statsforecast.models import AutoETS
from statsforecast.core import StatsForecast

/home/osprangers/Repositories/hierarchicalforecast/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
fcst = StatsForecast(models=[AutoETS(season_length=4, model='ZZA')], 
                     freq='QS', n_jobs=-1)
Y_hat_df = fcst.forecast(df=Y_train_df, h=8, fitted=True)
Y_fitted_df = fcst.forecast_fitted_values()

## 3. Reconcile forecasts

We reconcile using `BottomUp` as a baseline and `MinTrace` with every cross-sectional covariance method.

In [22]:
from hierarchicalforecast.methods import BottomUp, MinTrace, MiddleOutSparse, TopDownSparse
from hierarchicalforecast.core import HierarchicalReconciliation

In [52]:
reconcilers = [
    BottomUp(),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
    MinTrace(method='wls_var'),
    MinTrace(method='mint_shrink'),
    MinTrace(method='oasd'),
    TopDownSparse(method='forecast_proportions'),
    MiddleOutSparse(top_down_method='forecast_proportions', middle_level="Country/State"),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df, Y_df=Y_fitted_df, S_df=S_df, tags=tags)

The dataframe `Y_rec_df` contains the reconciled forecasts.

In [53]:
Y_rec_df.head()

,unique_id,ds,AutoETS,AutoETS/BottomUp,AutoETS/MinTrace_method-ols,AutoETS/MinTrace_method-wls_struct,AutoETS/MinTrace_method-wls_var,AutoETS/MinTrace_method-mint_shrink,AutoETS/MinTrace_method-oasd,AutoETS/TopDownSparse_method-forecast_proportions,AutoETS/MiddleOutSparse_middle_level-Country/State_top_down_method-forecast_proportions
0,Australia,2016-01-01,25990.068004,24381.672903,25932.138903,25202.989124,24859.505635,25101.233000,25968.413835,25990.068004,25636.333477
1,Australia,2016-04-01,24458.490282,22903.194015,24397.672041,23686.217991,23356.894384,23589.463536,23890.922505,24458.490282,24077.306104
2,Australia,2016-07-01,23974.055984,22411.401316,23907.281612,23180.656764,22851.109916,23085.887330,23511.477585,23974.055984,23544.178709
3,Australia,2016-10-01,24563.454495,23127.009693,24506.907660,23848.077529,23544.746803,23769.151444,24189.969280,24563.454495,24208.863105
4,Australia,2017-01-01,25990.068004,24518.047370,25933.168139,25253.667809,24940.822414,25168.631639,26499.529503,25990.068004,25636.333477


## 4. Evaluation 

The `HierarchicalForecast` package includes an `evaluate` function to evaluate the different hierarchies and also is capable of compute scaled metrics compared to a benchmark model.

In [54]:
from hierarchicalforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mase
from functools import partial

In [55]:
eval_tags = {}
eval_tags['Total'] = tags['Country']
# eval_tags['Purpose'] = tags['Country/Purpose']
eval_tags['State'] = tags['Country/State']
eval_tags['Regions'] = tags['Country/State/Region']
eval_tags['Bottom'] = tags['Country/State/Region/Purpose']

df = Y_rec_df.merge(Y_test_df, on=['unique_id', 'ds'])
evaluation = evaluate(df = df,
                      tags = eval_tags,
                      train_df = Y_train_df,
                      metrics = [rmse,
                                 partial(mase, seasonality=4)])

evaluation.columns = [
    'level', 'metric', 'Base', 'BottomUp',
    'OLS', 'WLS(struct)', 'WLS(var)',
    'Shrinkage', 'OASD', 'TopDownSparse', 'MiddleOutSparse'
]
numeric_cols = evaluation.select_dtypes(include="number").columns
evaluation[numeric_cols] = evaluation[numeric_cols].map('{:.2f}'.format).astype(np.float64)

### RMSE

RMSE across hierarchy levels for each covariance method.

In [56]:
evaluation.query('metric == "rmse"')

,level,metric,Base,BottomUp,OLS,WLS(struct),WLS(var),Shrinkage,OASD,TopDownSparse,MiddleOutSparse
0,Total,rmse,1743.29,3029.02,1791.21,2366.73,2644.95,2439.21,1737.89,1743.29,2059.19
2,State,rmse,308.15,413.44,286.47,341.48,371.17,350.44,400.43,278.57,308.15
4,Regions,rmse,51.66,55.13,46.35,49.47,51.04,49.34,67.92,43.95,46.14
6,Bottom,rmse,19.37,19.37,18.31,18.57,18.59,18.28,31.59,17.51,17.85
8,Overall,rmse,36.05,42.20,33.86,37.28,38.93,37.40,50.66,32.48,34.59


### MASE

MASE across hierarchy levels for each covariance method.

In [57]:
evaluation.query('metric == "mase"')

,level,metric,Base,BottomUp,OLS,WLS(struct),WLS(var),Shrinkage,OASD,TopDownSparse,MiddleOutSparse
1,Total,mase,1.59,3.16,1.64,2.36,2.70,2.45,1.65,1.59,1.96
3,State,mase,1.39,1.90,1.24,1.50,1.68,1.55,2.00,1.25,1.39
5,Regions,mase,1.12,1.19,0.99,1.05,1.11,1.07,1.60,0.99,1.03
7,Bottom,mase,0.98,0.98,1.07,0.99,0.96,0.95,1.77,0.94,0.95
9,Overall,mase,1.02,1.05,1.06,1.02,1.01,0.99,1.74,0.96,0.97


### Summary

Key observations:

- **Diagonal methods** (`ols`, `wls_struct`, `wls_var`) are fast and numerically stable. `ols` gives equal weight to all series; `wls_struct` weights by hierarchy structure; `wls_var` uses in-sample variance.
- **Shrinkage** (`shr`/`mint_shrink`) applies Ledoit-Wolf regularization to the sample covariance, enabling estimation even when the number of series exceeds the number of observations.
- **OASD** applies per-element Oracle Approximating Shrinkage to the correlation matrix, offering a middle ground between diagonal and full covariance methods.
- **Ill-conditioned methods**: `sam`/`mint_cov` and `bu` require more observations than series and are omitted here (n_series=425 > n_obs=72).

For the full list of available methods (including temporal and cross-temporal), see `list_covariance_methods()` from `hierarchicalforecast.covariance`.

### References
- [Wickramasuriya, S. L., Athanasopoulos, G., & Hyndman, R. J. (2019). "Optimal forecast reconciliation for hierarchical and grouped time series through trace minimization". Journal of the American Statistical Association, 114 , 804-819.](https://robjhyndman.com/publications/mint/)
- [Ledoit, O., & Wolf, M. (2004). "A well-conditioned estimator for large-dimensional covariance matrices". Journal of Multivariate Analysis, 88(2), 365-411.](https://doi.org/10.1016/S0047-259X(03)00096-4)
- [Schäfer, J., & Strimmer, K. (2005). "A Shrinkage Approach to Large-Scale Covariance Matrix Estimation and Implications for Functional Genomics". Statistical Applications in Genetics and Molecular Biology, 4(1).](https://doi.org/10.2202/1544-6115.1175)
- [Hyndman, R.J., & Athanasopoulos, G. (2021). "Forecasting: principles and practice, 3rd edition: Chapter 11: Forecasting hierarchical and grouped series.". OTexts: Melbourne, Australia.](https://otexts.com/fpp3/hierarchical.html)